# IFEST DAC 2026 — V16 BASELINE BEKU (v12A official-only, pretrained-free)
SVC(C=1.5, word24k+char32k) + LightGBM-450 (22 fitur evidence) + blend + overlay referensi train.
Tanpa external, tanpa pretrained/LLM/API, tanpa pseudo-label (`docs/rules.md`).

Status: baseline evaluasi kerja (nested BERSIH 0.6077, split identik v15 0.5955). BUKAN klaim vs
V12A-external / top-5 / 0.99745. Diagnostik referensi (LOO 4534 @ 0.998897) dilaporkan TERPISAH.


In [2]:
import csv, re, gc, math, json, time, random
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np

SEED = 20260911
random.seed(SEED); np.random.seed(SEED)

def find_csv(name):
    cands = [Path("data") / name, Path.cwd() / "data" / name]
    try:
        kg = Path("/kaggle/input")
        if kg.exists(): cands += sorted(kg.rglob(name))
    except Exception: pass
    for p in cands:
        if p.exists(): return p
    raise FileNotFoundError(name)

TRAIN_P = find_csv("train.csv"); TEST_P = find_csv("test.csv"); SMP_P = find_csv("sample_submission.csv")
try:
    OUT = Path("/kaggle/working/results_v16_nb"); OUT.mkdir(parents=True, exist_ok=True)
except Exception:
    OUT = Path("results/v16_notebook"); OUT.mkdir(parents=True, exist_ok=True)

rows = []
with open(TRAIN_P, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append([r["id"], r["title"] or "", r["content"] or "", int(r["label"])])
y = np.array([r[3] for r in rows])
assert len(rows) == 14397, len(rows)
print("train ok:", dict(Counter(y.tolist())), "out:", OUT, flush=True)

_re = re.compile(r"[^0-9a-zA-Z]+")
def norm_text(s): return " ".join(_re.sub(" ", str(s).lower()).split())
def toks(s): return norm_text(s).split()
NT = [norm_text(t) for _, t, _, _ in rows]; NC = [norm_text(c) for _, _, c, _ in rows]
parent = list(range(len(rows)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra
st_, sc_ = {}, {}
for i, (t, c) in enumerate(zip(NT, NC)):
    if t in st_: union(i, st_[t])
    else: st_[t] = i
    if c in sc_: union(i, sc_[c])
    else: sc_[c] = i
groups = np.array([find(i) for i in range(len(rows))])
print("grup DSU:", len(set(groups.tolist())), flush=True)


train ok: {0: 1437, 1: 12960} out: results/v16_notebook
grup DSU: 11526



In [3]:
# Diagnostik referensi LOO train-only (query dikeluarkan). TERPISAH dari OOF.
pair_counts = defaultdict(lambda: [0, 0]); cont_pos = defaultdict(Counter)
for t, c, yy in zip(NT, NC, y):
    pair_counts[(t, c)][int(yy)] += 1
    if int(yy) == 1: cont_pos[c][t] += 1
loo_n = loo_ok = 0
for t, c, yy in zip(NT, NC, y):
    a = pair_counts[(t, c)].copy(); a[int(yy)] -= 1
    p = None
    if sum(a) > 0 and (a[0] == 0 or a[1] == 0): p = 1 if a[1] > 0 else 0
    if p is None:
        ctr = cont_pos[c].copy()
        if int(yy) == 1:
            ctr[t] -= 1
            if ctr[t] <= 0: del ctr[t]
        if len(ctr) == 1: p = int(t == next(iter(ctr)))
    if p is not None:
        loo_n += 1; loo_ok += int(p == int(yy))
print("REF-LOO eligible=%d acc=%.6f (diagnostik komponen, BUKAN OOF)" % (loo_n, loo_ok / max(1, loo_n)), flush=True)


REF-LOO eligible=4534 acc=0.998897 (diagnostik komponen, BUKAN OOF)



In [4]:
MONTHS = {"januari","februari","maret","april","mei","juni","juli","agustus","september","oktober","november","desember","jan","feb","mar","apr","jun","jul","agu","sep","okt","nov","des"}
NEGS = {"tidak","tak","bukan","bukanlah","jangan","belum","tanpa","batal","gagal","hoaks","hoax","bohong","palsu","keliru","tunda","urung"}
def ngrams(x, n): return {" ".join(x[i:i+n]) for i in range(max(0, len(x)-n+1))}
def chargrams(s, n=3):
    s = norm_text(s).replace(" ", "_")
    return {s[i:i+n] for i in range(max(0, len(s)-n+1))}
def numbers(s): return re.findall(r"\b\d+(?:[.,]\d+)?\b", str(s).lower())
def proper_tokens(s): return [x.lower() for x in re.findall(r"\b[A-Z][A-Za-z]{2,}\b", str(s))]
def best_window(tt, ct, win=100, stride=70):
    if not ct: return [], 0.0, 0
    denom = len(set(tt)) or 1.0
    best, bs, bi = [], -1.0, 0
    for st in range(0, max(1, len(ct)), stride):
        ch = ct[st:st+win]
        if not ch: break
        ss = set(ch); sc_ = sum(1 for w in set(tt) if w in ss) / denom
        if sc_ > bs: best, bs, bi = ch, sc_, st
        if st + win >= len(ct): break
    return best, max(bs, 0.0), bi
def row_features(title, content):
    tt = toks(title); ct = toks(content); ts = set(tt); cs = set(ct); lead = ct[:100]
    lset = set(lead); bw, bscore, bpos = best_window(tt, ct); bset = set(bw)
    denom = max(1, len(ts))
    big = ngrams(tt, 2); tri = ngrams(tt, 3); c_norm = " ".join(ct)
    num_t = numbers(title); num_c = set(numbers(content))
    neg_t = [w for w in tt if w in NEGS]; neg_c = set(w for w in ct if w in NEGS)
    mon_t = [w for w in tt if w in MONTHS]; mon_c = set(w for w in ct if w in MONTHS)
    prop = proper_tokens(title); prop_c = set(x.lower() for x in re.findall(r"\b[A-Z][A-Za-z]{2,}\b", str(content)))
    cg = chargrams(title, 3); cc = chargrams(content, 3)
    d = {
        "title_words": len(tt), "content_words": len(ct), "log_content_words": math.log1p(len(ct)),
        "whole_token_cov": sum(w in cs for w in ts) / denom, "lead_token_cov": sum(w in lset for w in ts) / denom,
        "best_token_cov": sum(w in bset for w in ts) / denom,
        "idf_whole_cov": sum(w in cs for w in ts) / denom, "idf_lead_cov": sum(w in lset for w in ts) / denom,
        "idf_best_cov": bscore, "best_window_relpos": bpos / max(1, len(ct)),
        "bigram_cov": sum(g in c_norm for g in big) / max(1, len(big)),
        "trigram_cov": sum(g in c_norm for g in tri) / max(1, len(tri)),
        "char3_cov": len(cg & cc) / max(1, len(cg)),
        "num_count": len(num_t), "num_supported_frac": sum(x in num_c for x in num_t) / max(1, len(num_t)),
        "num_missing": sum(x not in num_c for x in num_t),
        "neg_count": len(neg_t), "neg_missing": sum(x not in neg_c for x in neg_t),
        "month_count": len(mon_t), "month_missing": sum(x not in mon_c for x in mon_t),
        "proper_count": len(prop), "proper_supported_frac": sum(x in prop_c for x in prop) / max(1, len(prop)),
        "title_content_len_ratio": len(tt) / max(1, len(ct)),
    }
    miss = [w for w in tt if w not in cs][:20]; pres = [w for w in tt if w in cs][:20]
    pair = ("TITLE " + " ".join(tt) + " LEAD " + " ".join(lead[:100]) + " EVIDENCE " + " ".join(bw[:100]) +
            " MISSING " + " ".join("miss_" + w for w in miss) +
            " SUPPORTED " + " ".join("hit_" + w for w in pres))
    return d, pair

import pandas as pd
FROWS, PTEXT = [], []
for _, t, x, _ in rows:
    v, s = row_features(t, x); FROWS.append(v); PTEXT.append(s)
DENSE = pd.DataFrame(FROWS).astype(np.float32)
PTEXT = np.asarray(PTEXT, dtype=object)
print("fitur 22-dim ok:", DENSE.shape, flush=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from scipy.special import expit
from scipy import sparse as _sp
from lightgbm import LGBMClassifier
def sample_weight(ytr):
    ytr = np.asarray(ytr, int); n = len(ytr); c = np.bincount(ytr, minlength=2)
    cw = np.array([n / (2 * max(1, c[0])), n / (2 * max(1, c[1]))], dtype=np.float32)
    return 8.0 * cw[ytr]
def make_vecs(ft):
    w = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=.997,
                        max_features=24000, sublinear_tf=True, dtype=np.float32)
    ch = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=.997,
                         max_features=32000, sublinear_tf=True, dtype=np.float32)
    w.fit(ft); ch.fit(ft)
    return w, ch
def sp_t(w, ch, txt):
    return _sp.hstack([w.transform(txt), ch.transform(txt)], format="csr", dtype=np.float32)
def new_lgb(seed):
    return LGBMClassifier(objective="binary", n_estimators=450, learning_rate=.03, num_leaves=31,
                          min_child_samples=40, subsample=.90, colsample_bytree=.90,
                          reg_alpha=.2, reg_lambda=2.0, random_state=seed, n_jobs=-1, verbosity=-1)
def fit_bases(fi_idx, seed):
    fi_idx = np.asarray(fi_idx)
    w, ch = make_vecs(PTEXT[fi_idx])
    Xr = sp_t(w, ch, PTEXT[fi_idx]); yr = y[fi_idx]; wr = sample_weight(yr)
    svc = LinearSVC(C=1.5, dual="auto", max_iter=5000, random_state=seed).fit(Xr, yr, sample_weight=wr)
    lgb = new_lgb(seed).fit(DENSE.iloc[fi_idx], yr, sample_weight=wr)
    return (w, ch, svc, lgb)
def pred_bases(bundle, idx):
    w, ch, svc, lgb = bundle; idx = np.asarray(idx)
    ss = svc.decision_function(sp_t(w, ch, PTEXT[idx])).astype(np.float32)
    lp = lgb.predict_proba(DENSE.iloc[idx])[:, 1].astype(np.float32)
    return ss, lp
print("builders ok", flush=True)


fitur 22-dim ok: (14397, 23)
builders ok



In [5]:
# Nested BERSIH: outer 5-fold; inner 2-fold pooled utk alpha+threshold; nilai outer-val.
ALPHAS = np.linspace(0, 1, 11); THR = np.arange(0.05, 0.951, 0.01)
def blend(ss, lp, a): return a * expit(ss) + (1 - a) * lp
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
rec = {k: [] for k in ("svc", "lgb", "blend")}; pols = []
t0 = time.time()
for fi, (tro, vao) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
    gtro = groups[tro]
    pos = {idx: k for k, idx in enumerate(tro.tolist())}
    i_sp = np.zeros(len(tro), np.float32); i_lp = np.zeros(len(tro), np.float32)
    for a, b in StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=100 + fi).split(np.zeros(len(tro)), y[tro], gtro):
        itr, iva = tro[a], tro[b]
        bundle = fit_bases(itr, SEED + fi)
        li = [pos[i] for i in iva.tolist()]
        ss, lp = pred_bases(bundle, iva)
        i_sp[li] = ss; i_lp[li] = lp
        del bundle; gc.collect()
    thr_svc, _ = max(((float(t), float(f1_score(y[tro], (i_sp >= t).astype(int), average="macro", zero_division=0))) for t in np.arange(-2, 2.01, 0.05)), key=lambda x: x[1])
    thr_lgb, _ = max(((float(t), float(f1_score(y[tro], (i_lp >= t).astype(int), average="macro", zero_division=0))) for t in THR), key=lambda x: x[1])
    ba, bt, bf = 0.5, 0.5, -1.0
    for a in ALPHAS:
        s = blend(i_sp, i_lp, float(a))
        for t in THR:
            f = f1_score(y[tro], (s >= t).astype(int), average="macro", zero_division=0)
            if f > bf: bf, ba, bt = f, float(a), float(t)
    bundle = fit_bases(tro, SEED + fi)
    ss_o, lp_o = pred_bases(bundle, vao)
    del bundle; gc.collect()
    for name, s, th in (("svc", ss_o, thr_svc), ("lgb", lp_o, thr_lgb), ("blend", blend(ss_o, lp_o, ba), bt)):
        rec[name].append(float(f1_score(y[vao], (s >= th).astype(int), average="macro", zero_division=0)))
    pols.append([ba, bt])
    print("fold %d svc=%.4f lgb=%.4f blend=%.4f (a=%.1f t=%.2f) (%ds)" % (
        fi, rec["svc"][-1], rec["lgb"][-1], rec["blend"][-1], ba, bt, time.time() - t0), flush=True)
for k in ("svc", "lgb", "blend"):
    print("CLEAN %s=%.4f %s" % (k, float(np.mean(rec[k])), [round(m, 4) for m in rec[k]]), flush=True)
with open(OUT / "metrics_nested.json", "w", encoding="utf-8") as f:
    json.dump({"svc": rec["svc"], "lgb": rec["lgb"], "blend": rec["blend"], "policies": pols}, f, indent=2)
print("nested saved", flush=True)


fold 0 svc=0.5356 lgb=0.6102 blend=0.5856 (a=0.6 t=0.57) (102s)
fold 1 svc=0.5705 lgb=0.6098 blend=0.6228 (a=0.5 t=0.50) (213s)
fold 2 svc=0.5656 lgb=0.6095 blend=0.6183 (a=0.5 t=0.54) (313s)
fold 3 svc=0.5785 lgb=0.6072 blend=0.6246 (a=0.5 t=0.55) (428s)
fold 4 svc=0.5564 lgb=0.5922 blend=0.5871 (a=0.5 t=0.55) (542s)
CLEAN svc=0.5613 [0.5356, 0.5705, 0.5656, 0.5785, 0.5564]
CLEAN lgb=0.6058 [0.6102, 0.6098, 0.6095, 0.6072, 0.5922]
CLEAN blend=0.6077 [0.5856, 0.6228, 0.6183, 0.6246, 0.5871]
nested saved



In [6]:
# Fitting FINAL + inferensi test SEKALI (model/policy beku dulu).
# Policy final = median policy 5 fold nested. Overlay referensi dari FULL train.
FA = float(np.median([p[0] for p in pols])); FT = float(np.median([p[1] for p in pols]))
print("frozen policy: alpha=%.2f threshold=%.2f" % (FA, FT), flush=True)
word, char, svc, lgb = fit_bases(np.arange(len(rows)), SEED + 900)
pair_map, canonical = {}, {}
tmp_p, tmp_c = defaultdict(set), defaultdict(set)
for t, c, yy in zip(NT, NC, y):
    tmp_p[(t, c)].add(int(yy))
    if int(yy) == 1: tmp_c[c].add(t)
for k, v in tmp_p.items():
    if len(v) == 1: pair_map[k] = next(iter(v))
for k, v in tmp_c.items():
    if len(v) == 1: canonical[k] = next(iter(v))
print("reference maps: pairs=%d canonical=%d" % (len(pair_map), len(canonical)), flush=True)

test_rows = []
with open(TEST_P, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        test_rows.append([r["id"], r["title"] or "", r["content"] or ""])
assert len(test_rows) == 3603, len(test_rows)
TF, TT = [], []
for _, t, x in test_rows:
    v, s = row_features(t, x); TF.append(v); TT.append(s)
import pandas as pd
TDF = pd.DataFrame(TF).astype(np.float32)
Xt = sp_t(word, char, np.asarray(TT, dtype=object))
ss = svc.decision_function(Xt).astype(np.float32)
lp = lgb.predict_proba(TDF)[:, 1].astype(np.float32)
score = FA * expit(ss) + (1 - FA) * lp
pred = (score >= FT).astype(int)
n_ref = 0
for i, (_, t, x) in enumerate(test_rows):
    nt, nc = norm_text(t), norm_text(x)
    if (nt, nc) in pair_map: pred[i] = pair_map[(nt, nc)]; n_ref += 1
    elif nc in canonical: pred[i] = int(nt == canonical[nc]); n_ref += 1
print("reference-overlay menyala di %d/3603 test" % n_ref, flush=True)
with open(OUT / "submission.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "label"])
    for (rid, _, _), p in zip(test_rows, pred):
        w.writerow([rid, int(p)])
with open(OUT / "manifest.json", "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "policy": [FA, FT], "ref_hits": int(n_ref),
               "towers": "SVC-C1.5-56k + LGBM-450-22feat + blend + ref-overlay",
               "external": "none", "pretrained": "none",
               "rules": "official-only, test-sealed, no-pretrained"}, f, indent=2)
print("submission + manifest saved", flush=True)


frozen policy: alpha=0.50 threshold=0.55
reference maps: pairs=12587 canonical=11138
reference-overlay menyala di 1139/3603 test
submission + manifest saved



In [7]:
smp = list(csv.DictReader(open(SMP_P, encoding="utf-8")))
sub = list(csv.DictReader(open(OUT / "submission.csv", encoding="utf-8")))
assert len(sub) == 3603 and len(smp) == 3603
assert [r["id"] for r in sub] == [r["id"] for r in smp]
assert set(r["label"] for r in sub) <= {"0", "1"}
print("submission VALID:", dict(Counter(r["label"] for r in sub)), flush=True)
print("VERDICT: baseline beku v12A-official. Bukan klaim vs external/top-5/0.99745.", flush=True)


submission VALID: {'0': 640, '1': 2963}
VERDICT: baseline beku v12A-official. Bukan klaim vs external/top-5/0.99745.

